# SmolVLM-256M — Monarch Linear + MonarchAttention + Fine-tuning Pipeline

**목표**: SmolVLM-256M의 linear projection을 Monarch 구조로 교체하고,  
Attention을 MonarchAttention으로 근사한 뒤 fine-tuning으로 품질을 회복한다.

**파이프라인**
```
0. 환경 확인
1. 모델 로드 & 베이스라인 inference
2. 레이어 분석 — Monarch 변환 대상 선정
3. MonarchLinear 구현
4. MonarchAttention 구현  (Q/K/V/O projection → Monarch)
5. 레이어 교체 (Dense → Monarch)
6. Fine-tuning
7. 사후 검증 (레이어 품질 + regression + 압축률 요약)
```

**모델 구조 요약 (SmolVLM-256M-Instruct)**
| 모듈 | 내용 |
|---|---|
| `model.vision_model` | SigLIP-B/16 (93M) — ViT encoder |
| `model.connector`   | MLP projection (12288→576) |
| `model.text_model`  | SmolLM2-135M — causal LM decoder |
| `lm_head`           | 576→49280 |


## 0. 환경 확인

In [1]:
import subprocess, sys, os, platform
import torch

print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

import transformers, PIL
print("transformers:", transformers.__version__)
print("Pillow      :", PIL.__version__)

# OpenBLAS 링크 확인
cfg = torch.__config__.show()
if "OpenBLAS" in cfg or "openblas" in cfg.lower():
    print("BLAS    : OpenBLAS ✓")
else:
    print("BLAS    : OpenBLAS 미감지 — RPi5에서는 수동 링크 권장")
    print("          sudo apt install libopenblas-dev")
    print("          pip install torch --extra-index-url ... (BLAS-linked wheel)")


Python  : 3.10.20
Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39
PyTorch : 2.12.1+cu130
CUDA    : True
GPU     : NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
VRAM    : 102.0 GB


/home/24_ghangmin/miniconda3/envs/rpi5/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.12.1
Pillow      : 12.2.0
BLAS    : OpenBLAS 미감지 — RPi5에서는 수동 링크 권장
          sudo apt install libopenblas-dev
          pip install torch --extra-index-url ... (BLAS-linked wheel)


## 1. 모델 로드 & 베이스라인 Inference

In [2]:
import gc, time, copy
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
import transformers
# Accelerate dispatch_model is intentionally not used for training here;
# SmolVLM/Idefics3 image preprocessing triggered CUDA device-side asserts when sharded.

# ── 설정 ──────────────────────────────────────────────────────────
MODEL_ID    = "HuggingFaceTB/SmolVLM-256M-Instruct"
IMAGE_PATH  = "/users_gpu/24_ghangmin/CourseWork/2026_01_semi/project/vlm/Cat03.jpg"
WORKSPACE   = Path("/users_gpu/24_ghangmin/CourseWork/2026_01_semi/project/vlm")
CKPT_DIR    = WORKSPACE / "monarch_checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

# Use one clean GPU for training.  Multi-GPU Accelerate dispatch was tested but
# triggered a CUDA device-side assert inside Idefics3.get_image_features on this
# Torch/Transformers stack.  Keep conversion cached on CPU, then train on cuda:2.
GPU_IDS = [2]
USE_MULTI_GPU = False
PRIMARY_GPU = GPU_IDS[0] if torch.cuda.is_available() else None
if torch.cuda.is_available():
    torch.cuda.set_device(PRIMARY_GPU)
DEVICE = f"cuda:{PRIMARY_GPU}" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
MODEL_DEVICE_MAP = None

# ── 버전별 모델 클래스 ─────────────────────────────────────────────
_tv = tuple(int(x) for x in transformers.__version__.split(".")[:2])
if _tv >= (5, 0):
    from transformers import AutoModelForImageTextToText as _ModelCls
else:
    from transformers import AutoModelForVision2Seq as _ModelCls
from transformers import AutoProcessor

def model_input_device(mdl) -> torch.device:
    """Return the device where token/image inputs should be placed."""
    hf_map = getattr(mdl, "hf_device_map", None)
    if isinstance(hf_map, dict):
        cuda_devs = []
        for dev in hf_map.values():
            if isinstance(dev, int):
                cuda_devs.append(torch.device(f"cuda:{dev}"))
            elif isinstance(dev, str) and dev.startswith("cuda"):
                cuda_devs.append(torch.device(dev))
        if cuda_devs:
            return sorted(cuda_devs, key=lambda d: d.index or 0)[0]
    return next(mdl.parameters()).device

def print_cuda_memory(prefix="CUDA memory"):
    if not torch.cuda.is_available():
        return
    print(prefix)
    for gid in GPU_IDS:
        free, total = torch.cuda.mem_get_info(gid)
        used = total - free
        print(f"  cuda:{gid}: used={used/1e9:.2f} GB free={free/1e9:.2f} GB total={total/1e9:.2f} GB")

def dispatch_for_training(mdl):
    """Move the already-mutated Monarch model onto the selected training GPU once."""
    if USE_MULTI_GPU:
        raise RuntimeError("USE_MULTI_GPU is disabled for SmolVLM training because Accelerate sharding triggered CUDA device-side asserts in get_image_features.")
    print(f"Moving converted model to {DEVICE}")
    return mdl.to(DEVICE)

print(f"device/input : {DEVICE}")
print(f"multi_gpu    : {USE_MULTI_GPU}, GPU_IDS={GPU_IDS}")
print("device_map   : disabled; single-GPU training avoids Idefics3 image-path CUDA assert")
print(f"dtype        : {DTYPE}")
print(f"model        : {MODEL_ID}")
print_cuda_memory("initial CUDA memory")


device/input : cuda:2
multi_gpu    : False, GPU_IDS=[2]
device_map   : disabled; single-GPU training avoids Idefics3 image-path CUDA assert
dtype        : torch.bfloat16
model        : HuggingFaceTB/SmolVLM-256M-Instruct
initial CUDA memory
  cuda:2: used=0.60 GB free=101.37 GB total=101.97 GB


In [3]:
# 모델 로드
# Load on CPU first.  Monarch replacement happens before Accelerate dispatch;
# this avoids stale dispatch hooks when many nn.Linear modules are swapped out.
gc.collect()
if torch.cuda.is_available():
    for gid in GPU_IDS:
        with torch.cuda.device(gid):
            torch.cuda.empty_cache()

t0 = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = _ModelCls.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    _attn_implementation="eager",   # RPi5: eager. GPU 있으면 "flash_attention_2" 가능
).to("cpu")
model.eval()
print(f"CPU 로드 완료: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params, "
      f"{time.perf_counter()-t0:.1f}s")
print("hf_device_map:", getattr(model, "hf_device_map", None))
print("current model device:", next(model.parameters()).device)
print_cuda_memory("after CPU model load")


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2046.35it/s]


CPU 로드 완료: 256.5M params, 5.0s
hf_device_map: None
current model device: cpu
after CPU model load
  cuda:2: used=0.60 GB free=101.37 GB total=101.97 GB


In [4]:
# 베이스라인 Inference
def smolvlm_query(mdl, proc, image: Image.Image, question: str,
                  max_new_tokens: int = 128) -> str:
    messages = [{
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": question}],
    }]
    prompt = proc.apply_chat_template(messages, add_generation_prompt=True)
    target_device = model_input_device(mdl)
    inputs = proc(text=prompt, images=[image], return_tensors="pt").to(target_device)
    with torch.inference_mode():
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[:, inputs["input_ids"].shape[-1]:]
    return proc.batch_decode(gen, skip_special_tokens=True)[0].strip()

test_image = Image.open(IMAGE_PATH).convert("RGB")
print("Baseline generation is skipped before replacement because the model is still on CPU.")
print("It will be checked after the Monarch model is dispatched to cuda:1/cuda:2.")


Baseline generation is skipped before replacement because the model is still on CPU.
It will be checked after the Monarch model is dispatched to cuda:1/cuda:2.


## 2. 레이어 분석 — Monarch 변환 대상 선정

SmolVLM-256M 레이어 패밀리:

| 패밀리 | 모듈 경로 패턴 | 설명 |
|---|---|---|
| `siglip_attn` | `vision_model...attn` | SigLIP ViT attention QKV/proj |
| `siglip_ffn`  | `vision_model...mlp`  | SigLIP ViT FFN (fc1/fc2) |
| `connector`   | `connector`           | MLP projector (12288→576) |
| `text_attn`   | `text_model...attn`   | SmolLM2 attention |
| `text_ffn`    | `text_model...mlp`    | SmolLM2 FFN |

**변환 단계 (CONVERSION_STAGE)**:
- Stage 1 — `siglip_ffn` 만 (가장 안전, 속도 이득 큼)
- Stage 2 — + `siglip_attn`
- Stage 3 — + `text_ffn`
- Stage 4 — + `text_attn` (가장 적극적)


In [5]:
from collections import defaultdict

def audit_linear_layers(mdl):
    records = []
    for name, module in mdl.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        rows, cols = module.weight.shape  # (out, in)
        n = name.lower()
        if "vision_model" in n and ("self_attn" in n or "attn" in n) and "mlp" not in n:
            family = "siglip_attn"
        elif "vision_model" in n and ("mlp" in n or "fc1" in n or "fc2" in n):
            family = "siglip_ffn"
        elif "connector" in n:
            family = "connector"
        elif "text_model" in n and ("self_attn" in n or "attn" in n) and "mlp" not in n:
            family = "text_attn"
        elif "text_model" in n and "mlp" in n:
            family = "text_ffn"
        elif "lm_head" in n:
            family = "lm_head"
        else:
            family = "other"
        monarch_ok = (rows >= 64 and cols >= 64
                      and rows % 2 == 0 and cols % 2 == 0)
        records.append({
            "name": name, "family": family,
            "out": rows, "in": cols,
            "params": rows * cols,
            "bias": module.bias is not None,
            "dtype": str(module.weight.dtype),
            "monarch_candidate": monarch_ok,
        })
    return records

layer_info = audit_linear_layers(model)

fam_summary = defaultdict(lambda: {"total": 0, "candidates": 0, "params": 0})
for r in layer_info:
    fam_summary[r["family"]]["total"] += 1
    fam_summary[r["family"]]["params"] += r["params"]
    if r["monarch_candidate"]:
        fam_summary[r["family"]]["candidates"] += 1

print(f"전체 nn.Linear: {len(layer_info)}개")
print(f"{'패밀리':15s} {'레이어':>6} {'후보':>6} {'파라미터':>12}")
print("-" * 45)
for fam, s in sorted(fam_summary.items()):
    print(f"{fam:15s} {s['total']:>6} {s['candidates']:>6} {s['params']/1e6:>10.2f}M")


전체 nn.Linear: 284개
패밀리                레이어     후보         파라미터
---------------------------------------------
connector            1      1       7.08M
lm_head              1      1      28.39M
siglip_attn         48     48      28.31M
siglip_ffn          24     24      56.62M
text_attn          120    120      26.54M
text_ffn            90     90      79.63M


In [6]:
import json
# 변환 대상 선정
# Modes:
# - fc1_vision_text: user-requested fc1-like setting. Converts vision MLP fc1
#   and text MLP expansion projections gate_proj/up_proj. Excludes vision fc2,
#   text down_proj, attention, connector, and lm_head.
# - all_ffn_vision_text: aggressive FFN setting. Converts all vision MLP fc1/fc2
#   and all text MLP gate/up/down layers. Excludes attention and connector.
# - measured_safe: use smoke_monarch/layer_screen_results*.json and only convert
#   layers whose dense-vs-Monarch random-input fit is acceptable.
# - five_layer_sanity: older hand-picked sanity set.
# - stage: broad family-level replacement; not recommended until smoke tests pass.
REPLACEMENT_MODE = "fc1_vision_text"
CONVERSION_STAGE = 1   # used only when REPLACEMENT_MODE == "stage"

TARGET_FAMILIES = {
    1: ["siglip_ffn"],
    2: ["siglip_ffn", "siglip_attn"],
    3: ["siglip_ffn", "siglip_attn", "text_ffn"],
    4: ["siglip_ffn", "siglip_attn", "text_ffn", "text_attn"],
}[CONVERSION_STAGE]

MEASURED_SAFE_JSON = WORKSPACE / "smoke_monarch" / "layer_screen_results_1p5x_vision_all.json"
SAFE_COS_THRESHOLD = 0.95
SAFE_REL_ERROR_THRESHOLD = 0.35
SAFE_ALLOWED_FAMILIES = {"siglip_attn", "siglip_ffn"}
SAFE_MAX_LAYERS = 32

VISION_LAYER_IDS = range(12)   # 0 ~ 11
TEXT_LAYER_IDS   = range(30)   # 0 ~ 29

SANITY_LAYER_NAMES = {
    *{f"model.vision_model.encoder.layers.{i}.self_attn.q_proj" for i in VISION_LAYER_IDS},
    *{f"model.vision_model.encoder.layers.{i}.mlp.fc1" for i in VISION_LAYER_IDS},
    "model.connector.modality_projection.proj",
    *{f"model.text_model.layers.{i}.self_attn.q_proj" for i in TEXT_LAYER_IDS},
    *{f"model.text_model.layers.{i}.mlp.gate_proj" for i in TEXT_LAYER_IDS},
}

def _is_vision_fc1(name: str) -> bool:
    return name.startswith("model.vision_model.encoder.layers.") and name.endswith(".mlp.fc1")

def _is_text_fc1_like(name: str) -> bool:
    # SmolLM/SwiGLU MLP has no literal fc1. gate_proj and up_proj are the two
    # expansion-side projections; down_proj is the fc2-like contraction and is excluded.
    return name.startswith("model.text_model.layers.") and (name.endswith(".mlp.gate_proj") or name.endswith(".mlp.up_proj"))

if REPLACEMENT_MODE == "fc1_vision_text":
    TARGET_LAYERS = [
        r for r in layer_info
        if r["monarch_candidate"] and (_is_vision_fc1(r["name"]) or _is_text_fc1_like(r["name"]))
    ]
    SAFE_LAYER_METRICS = {}
    print("Replacement mode: fc1_vision_text")
    print("Includes: vision mlp.fc1 and text mlp gate_proj/up_proj. Excludes fc2/down_proj, attention, connector, lm_head.")
elif REPLACEMENT_MODE == "all_ffn_vision_text":
    TARGET_LAYERS = [
        r for r in layer_info
        if r["family"] in {"siglip_ffn", "text_ffn"} and r["monarch_candidate"]
    ]
    SAFE_LAYER_METRICS = {}
    print("Replacement mode: all_ffn_vision_text")
    print("Includes: vision MLP fc1/fc2 and text MLP gate/up/down. Excludes attention and connector.")
elif REPLACEMENT_MODE == "measured_safe":
    if not MEASURED_SAFE_JSON.exists():
        raise FileNotFoundError(
            f"Measured-safe layer screen missing: {MEASURED_SAFE_JSON}. "
            "Run: python smoke_monarch/run_layer_screen.py --device cuda:0"
        )
    screen_payload = json.loads(MEASURED_SAFE_JSON.read_text())
    safe_rows = [
        r for r in screen_payload["results"]
        if r.get("status") == "ok"
        and r.get("family") in SAFE_ALLOWED_FAMILIES
        and r.get("cosine", -1.0) >= SAFE_COS_THRESHOLD
        and r.get("rel_error", 1e9) <= SAFE_REL_ERROR_THRESHOLD
    ]
    safe_rows = sorted(safe_rows, key=lambda r: (-r["cosine"], r["rel_error"]))[:SAFE_MAX_LAYERS]
    SAFE_LAYER_METRICS = {r["name"]: r for r in safe_rows}
    safe_names = {r["name"] for r in safe_rows}
    TARGET_LAYERS = [r for r in layer_info if r["name"] in safe_names and r["monarch_candidate"]]
    print("Replacement mode: measured_safe")
    print(f"screen file: {MEASURED_SAFE_JSON}")
    print(f"thresholds: cosine>={SAFE_COS_THRESHOLD}, rel_error<={SAFE_REL_ERROR_THRESHOLD}")
    print(f"allowed families: {sorted(SAFE_ALLOWED_FAMILIES)}")
elif REPLACEMENT_MODE == "five_layer_sanity":
    TARGET_LAYERS = [r for r in layer_info if r["name"] in SANITY_LAYER_NAMES and r["monarch_candidate"]]
    SAFE_LAYER_METRICS = {}
    print("Replacement mode: expanded_sanity")
    print("This replaces selected sanity targets across vision layers 0~11 and text layers 0~29.")
elif REPLACEMENT_MODE == "stage":
    TARGET_LAYERS = [r for r in layer_info if r["family"] in TARGET_FAMILIES and r["monarch_candidate"]]
    SAFE_LAYER_METRICS = {}
    print(f"Replacement mode: stage {CONVERSION_STAGE}")
else:
    raise ValueError(f"unknown REPLACEMENT_MODE={REPLACEMENT_MODE!r}")

if not TARGET_LAYERS:
    raise RuntimeError("No Monarch target layers selected. Check REPLACEMENT_MODE or thresholds.")

target_params = sum(r["params"] for r in TARGET_LAYERS)
total_params  = sum(r["params"] for r in layer_info)

print(f"변환 대상: {len(TARGET_LAYERS)}개 레이어")
if REPLACEMENT_MODE == "stage":
    print(f"대상 패밀리: {TARGET_FAMILIES}")

print(f"변환 파라미터: {target_params/1e6:.1f}M / {total_params/1e6:.1f}M ({100*target_params/total_params:.1f}%)")

for r in TARGET_LAYERS:
    extra = ""
    if REPLACEMENT_MODE == "measured_safe":
        m = SAFE_LAYER_METRICS.get(r["name"], {})
        extra = f" cos={m.get('cosine', float('nan')):.4f} rel={m.get('rel_error', float('nan')):.4f}"
    print(f"  - {r['family']:12s} {r['name']} {r['out']}x{r['in']}{extra}")


Replacement mode: fc1_vision_text
Includes: vision mlp.fc1 and text mlp gate_proj/up_proj. Excludes fc2/down_proj, attention, connector, lm_head.
변환 대상: 72개 레이어
변환 파라미터: 81.4M / 226.6M (35.9%)
  - siglip_ffn   model.vision_model.encoder.layers.0.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.1.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.2.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.3.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.4.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.5.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.6.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.7.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.8.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.9.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.encoder.layers.10.mlp.fc1 3072x768
  - siglip_ffn   model.vision_model.

## 3. MonarchLinear 구현

In [7]:
import math
import torch.nn.functional as F

from monarch_paper import (
    MonarchRectangular,
    MonarchSquare,
    best_nblocks_for_ratio,
    compute_fit_metrics,
    structured_parameter_count,
)

class MonarchLinear(MonarchRectangular):
    """Compatibility wrapper around the paper-faithful Monarch operator."""

    @staticmethod
    def _solve_nblocks(in_f, out_f, target_cr):
        return best_nblocks_for_ratio(in_f, out_f, target_cr)

    def init_from_dense(self, weight: torch.Tensor):
        fitted = MonarchLinear.project_from_dense(
            weight, nblocks=self.nblocks, bias=self.bias is not None
        )
        with torch.no_grad():
            self.left.copy_(fitted.left)
            self.right.copy_(fitted.right)
            if self.bias is not None and fitted.bias is not None:
                self.bias.copy_(fitted.bias)
        return self

    @property
    def L(self):
        return self.left

    @property
    def R(self):
        return self.right

    @property
    def in_per_block(self):
        return self.in_block

    @property
    def out_per_block(self):
        return self.out_block

    @property
    def blk_rank(self):
        return self.nblocks

    def compression_ratio(self):
        return super().compression_ratio()


_sq = MonarchSquare(16, bias=True)
_x = torch.randn(2, 16)
_y = _sq(_x)
assert _y.shape == (2, 16)
print('paper-faithful Monarch module loaded')


paper-faithful Monarch module loaded


## 4. MonarchAttention 구현

기존 SigLIP / SmolLM2 Attention 모듈의 Q/K/V/O projection을  
`MonarchLinear`로 교체하는 wrapper.

> **설계 원칙**  
> Attention 로직(scaled dot-product)은 그대로 두고,  
> projection만 Monarch로 바꿔 drop-in 교체가 가능하게 한다.


In [8]:
def replace_proj_in_attn(attn_module: nn.Module,
                          proj_names: list[str],
                          target_compression: float = 2.0,
                          init_from_dense: bool = True,
                          verbose: bool = False) -> dict:
    results = {}
    for pname in proj_names:
        if not hasattr(attn_module, pname):
            if verbose:
                print(f"  [skip] {pname} — attribute 없음")
            continue
        orig = getattr(attn_module, pname)
        if not isinstance(orig, nn.Linear):
            if verbose:
                print(f"  [skip] {pname} — nn.Linear가 아님 ({type(orig).__name__})")
            continue
        nblocks = MonarchLinear._solve_nblocks(orig.in_features, orig.out_features, target_compression)
        try:
            ml = MonarchLinear.project_from_dense(
                orig.weight.data,
                nblocks=nblocks,
                bias=orig.bias.data if orig.bias is not None else None,
            )
            ml = ml.to(orig.weight.device).to(orig.weight.dtype)
            setattr(attn_module, pname, ml)
            results[pname] = ml.compression_ratio()
            if verbose:
                print(f"  [OK] {pname}: {orig.in_features}→{orig.out_features}, nblocks={nblocks}, CR={ml.compression_ratio():.2f}x")
        except Exception as e:
            if verbose:
                print(f"  [skip] {pname} — {e}")
    return results


def detect_attn_proj_names(attn_module: nn.Module) -> list[str]:
    candidates = ["q_proj", "k_proj", "v_proj", "out_proj",
                  "qkv_proj", "o_proj", "query", "key", "value", "dense"]
    return [n for n in candidates if isinstance(getattr(attn_module, n, None), nn.Linear)]


# Unit-test replacement on a deepcopy so the real model is not mutated before
# the selected replacement cells run.
_sample_attn = copy.deepcopy(model.model.vision_model.encoder.layers[0].self_attn).to("cpu")
_proj_names = detect_attn_proj_names(_sample_attn)
print(f"SigLIP attention projection 이름: {_proj_names}")
_results = replace_proj_in_attn(_sample_attn, _proj_names, target_compression=2.0, init_from_dense=True, verbose=True)
print(f"교체 결과: {_results}")
del _sample_attn
gc.collect()


SigLIP attention projection 이름: ['q_proj', 'k_proj', 'v_proj', 'out_proj']
  [OK] q_proj: 768→768, nblocks=192, CR=1.99x
  [OK] k_proj: 768→768, nblocks=192, CR=1.99x
  [OK] v_proj: 768→768, nblocks=192, CR=1.99x
  [OK] out_proj: 768→768, nblocks=192, CR=1.99x
교체 결과: {'q_proj': 1.9948051948051948, 'k_proj': 1.9948051948051948, 'v_proj': 1.9948051948051948, 'out_proj': 1.9948051948051948}


0

## 5. 레이어 교체 (Dense → Monarch)

In [9]:
# ── 설정 ──────────────────────────────────────────────────────────
TARGET_COMPRESSION = 1.5   # measured-safe smoke default; 2.0 all-layer caused severe quality loss
INIT_FROM_DENSE    = True
USE_CONVERTED_CACHE = True
FORCE_RECONVERT     = False  # set True only when changing Monarch implementation or target selection
if TARGET_COMPRESSION > 4.0:
    print(f"[warn] target_compression={TARGET_COMPRESSION} exceeds the paper-validated 2x-4x regime; treat as exploratory.")

import hashlib, json
_target_signature = json.dumps({
    "model_id": MODEL_ID,
    "replacement_mode": REPLACEMENT_MODE,
    "conversion_stage": CONVERSION_STAGE,
    "target_compression": TARGET_COMPRESSION,
    "target_layers": [r["name"] for r in TARGET_LAYERS],
    "attention_replacement": "exact_projection_v2",
    "target_selection_version": "fc1_vision_text_v1",
}, sort_keys=True)
CACHE_TAG = hashlib.sha1(_target_signature.encode()).hexdigest()[:12]
CONVERTED_CACHE_DIR = CKPT_DIR / f"converted_monarch_{CACHE_TAG}"
CONVERTED_STATE_PATH = CONVERTED_CACHE_DIR / "model_state_dict.pt"
CONVERTED_MANIFEST_PATH = CONVERTED_CACHE_DIR / "conversion_manifest.json"
print("converted cache:", CONVERTED_CACHE_DIR)
print("cache exists:", CONVERTED_STATE_PATH.exists(), "force_reconvert:", FORCE_RECONVERT)

# 원본 모델 복사 (사후 비교용). Do not deepcopy the sharded GPU model here:
# that can temporarily duplicate tensors on cuda:1/cuda:2. Load a CPU reference
# directly instead.
gc.collect()
if torch.cuda.is_available():
    for gid in GPU_IDS:
        with torch.cuda.device(gid):
            torch.cuda.empty_cache()
model_orig = _ModelCls.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    _attn_implementation="eager",
).to("cpu")
model_orig.eval()
gc.collect()
print("원본 모델 로드 완료 (CPU 보관)")
print_cuda_memory("after CPU reference load")


converted cache: /users_gpu/24_ghangmin/CourseWork/2026_01_semi/project/vlm/monarch_checkpoints/converted_monarch_b679c67f81f8
cache exists: True force_reconvert: False


Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2933.34it/s]


원본 모델 로드 완료 (CPU 보관)
after CPU reference load
  cuda:2: used=0.60 GB free=101.37 GB total=101.97 GB


In [10]:
def get_module(root, dotted_name: str) -> nn.Module:
    m = root
    for p in dotted_name.split('.'):
        m = getattr(m, p)
    return m

def set_module(root, dotted_name: str, new_mod: nn.Module):
    parts = dotted_name.split('.')
    parent = get_module(root, '.'.join(parts[:-1]))
    setattr(parent, parts[-1], new_mod)

def _monarch_record_from_module(name: str, ml: MonarchRectangular) -> dict:
    return {
        'name': name,
        'in': int(ml.in_features),
        'out': int(ml.out_features),
        'nblocks': int(ml.nblocks),
        'cr': float(ml.compression_ratio()),
    }

def rebuild_monarch_modules_from_state(mdl, state: dict) -> list[dict]:
    """Replace dense Linear modules with Monarch modules based on saved .left/.right keys."""
    rebuilt = []
    left_keys = sorted(k for k in state.keys() if k.endswith('.left'))
    for left_key in left_keys:
        name = left_key[:-5]
        right_key = name + '.right'
        if right_key not in state:
            raise KeyError(f"missing right factor for cached Monarch layer {name}")
        left = state[left_key]
        right = state[right_key]
        if left.ndim != 3 or right.ndim != 3:
            raise ValueError(f"bad cached factor shape for {name}: left={tuple(left.shape)} right={tuple(right.shape)}")
        nblocks = int(left.shape[0])
        out_features = int(left.shape[1] * nblocks)
        in_features = int(right.shape[2] * nblocks)
        has_bias = (name + '.bias') in state
        ml = MonarchLinear(in_features, out_features, nblocks=nblocks, bias=has_bias)
        ml = ml.to(device='cpu', dtype=left.dtype)
        set_module(mdl, name, ml)
        rebuilt.append(_monarch_record_from_module(name, ml))
    return rebuilt

def load_converted_cache_if_available(mdl) -> tuple[bool, list[dict], dict]:
    if not USE_CONVERTED_CACHE or FORCE_RECONVERT or not CONVERTED_STATE_PATH.exists():
        return False, [], {}
    print(f"Loading converted Monarch cache: {CONVERTED_STATE_PATH}")
    state = torch.load(CONVERTED_STATE_PATH, map_location='cpu')
    rebuilt = rebuild_monarch_modules_from_state(mdl, state)
    missing, unexpected = mdl.load_state_dict(state, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"cached state mismatch: missing={len(missing)} unexpected={len(unexpected)}")
    manifest = json.loads(CONVERTED_MANIFEST_PATH.read_text()) if CONVERTED_MANIFEST_PATH.exists() else {}
    print(f"Converted cache loaded: {len(rebuilt)} Monarch layers rebuilt; SVD conversion skipped")
    return True, rebuilt, manifest

def save_converted_cache(mdl, replaced_ffn_records, replaced_attn_records, all_replaced_records):
    if not USE_CONVERTED_CACHE:
        return
    CONVERTED_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Saving converted Monarch cache: {CONVERTED_STATE_PATH}")
    cpu_state = {k: v.detach().cpu() for k, v in mdl.state_dict().items()}
    torch.save(cpu_state, CONVERTED_STATE_PATH)
    manifest = {
        "model_id": MODEL_ID,
        "replacement_mode": REPLACEMENT_MODE,
        "conversion_stage": CONVERSION_STAGE,
        "target_compression": TARGET_COMPRESSION,
        "target_layer_count": len(TARGET_LAYERS),
        "target_signature_sha1": CACHE_TAG,
        "target_layers": [r["name"] for r in TARGET_LAYERS],
        "attention_replacement": "exact_projection_v2",
        "target_selection_version": "fc1_vision_text_v1",
        "replaced_ffn": replaced_ffn_records,
        "replaced_attn_detail": replaced_attn_records,
        "all_replaced": all_replaced_records,
    }
    CONVERTED_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
    print(f"Converted cache saved: {CONVERTED_CACHE_DIR}")

def replace_linear_layers(mdl, target_records: list[dict], compression: float = 2.0, init_from_dense: bool = True) -> list[dict]:
    replaced, failed = [], []
    for r in target_records:
        name = r['name']
        try:
            orig = get_module(mdl, name)
        except AttributeError:
            failed.append((name, 'AttributeError'))
            continue
        if not isinstance(orig, nn.Linear):
            failed.append((name, f'not Linear: {type(orig).__name__}'))
            continue
        try:
            nblocks = MonarchLinear._solve_nblocks(orig.in_features, orig.out_features, compression)
            ml = MonarchLinear.project_from_dense(
                orig.weight.data,
                nblocks=nblocks,
                bias=orig.bias.data if orig.bias is not None else None,
            )
            ml = ml.to(orig.weight.device).to(orig.weight.dtype)
            set_module(mdl, name, ml)
            replaced.append(_monarch_record_from_module(name, ml))
            del orig, ml
            gc.collect()
        except Exception as e:
            failed.append((name, str(e)))
    print(f"교체 완료: {len(replaced)}개 / 실패: {len(failed)}개")
    if failed:
        for n, reason in failed[:5]:
            print(f"  [실패] {n}: {reason}")
    return replaced

ffn_families = {"siglip_ffn", "connector", "text_ffn"}
attn_families = {"siglip_attn", "text_attn"}
ffn_targets = [r for r in TARGET_LAYERS if r['family'] in ffn_families]
attn_targets = [r for r in TARGET_LAYERS if r['family'] in attn_families]
print(f"FFN  교체 대상: {len(ffn_targets)}개")
print(f"Attn 교체 대상: {len(attn_targets)}개 레이어 (projection 단위)")

CONVERSION_CACHE_LOADED, _cached_rebuilt, _cache_manifest = load_converted_cache_if_available(model)
if CONVERSION_CACHE_LOADED:
    replaced_ffn = _cache_manifest.get('replaced_ffn', [])
    replaced_attn_detail = _cache_manifest.get('replaced_attn_detail', [])
    if not replaced_ffn and not replaced_attn_detail:
        # Manifest from older cache may be absent; fall back to rebuilt records as generic replacements.
        replaced_ffn = [r for r in _cached_rebuilt if any(r['name'] == t['name'] for t in ffn_targets)]
        replaced_attn_detail = []


FFN  교체 대상: 72개
Attn 교체 대상: 0개 레이어 (projection 단위)
Loading converted Monarch cache: /users_gpu/24_ghangmin/CourseWork/2026_01_semi/project/vlm/monarch_checkpoints/converted_monarch_b679c67f81f8/model_state_dict.pt
Converted cache loaded: 72 Monarch layers rebuilt; SVD conversion skipped


In [11]:
# ── FFN Linear 교체 ───────────────────────────────────────────────
if CONVERSION_CACHE_LOADED:
    print(f"FFN 교체: cache loaded, skipping SVD conversion ({len(replaced_ffn)} cached records)")
else:
    replaced_ffn = replace_linear_layers(
        model, ffn_targets,
        compression=TARGET_COMPRESSION,
        init_from_dense=INIT_FROM_DENSE,
    )


FFN 교체: cache loaded, skipping SVD conversion (72 cached records)


In [12]:
# ── Attention projection 교체 (exact measured-safe projection only) ───────────────
if CONVERSION_CACHE_LOADED:
    print(f"Attention projection 교체: cache loaded, skipping SVD conversion ({len(replaced_attn_detail)} cached records)")
else:
    replaced_attn_detail = []

    if attn_targets:
        for r in attn_targets:
            name = r['name']
            attn_path = '.'.join(name.split('.')[:-1])
            proj_name = name.split('.')[-1]
            try:
                attn_mod = get_module(model, attn_path)
            except AttributeError:
                print(f"[skip] {attn_path} — 모듈 없음")
                continue
            if proj_name not in detect_attn_proj_names(attn_mod):
                print(f"[skip] {name} — projection not detected as nn.Linear")
                continue
            orig = getattr(attn_mod, proj_name)
            if not isinstance(orig, nn.Linear):
                print(f"[skip] {name} — not Linear: {type(orig).__name__}")
                continue
            try:
                nblocks = MonarchLinear._solve_nblocks(orig.in_features, orig.out_features, TARGET_COMPRESSION)
                ml = MonarchLinear.project_from_dense(
                    orig.weight.data,
                    nblocks=nblocks,
                    bias=orig.bias.data if orig.bias is not None else None,
                )
                ml = ml.to(orig.weight.device).to(orig.weight.dtype)
                setattr(attn_mod, proj_name, ml)
                replaced_attn_detail.append({
                    'attn_path': attn_path,
                    'proj': proj_name,
                    'name': name,
                    'in': orig.in_features,
                    'out': orig.out_features,
                    'nblocks': nblocks,
                    'cr': ml.compression_ratio(),
                })
                del orig, ml
                gc.collect()
            except Exception as e:
                print(f"[skip] {name} — {e}")
        print(f"Attention projection 교체: {len(replaced_attn_detail)}개")
    else:
        print("Attention 교체 없음")


Attention projection 교체: cache loaded, skipping SVD conversion (0 cached records)


In [13]:
# ── 교체 후 압축률 요약 ───────────────────────────────────────────
all_replaced = replaced_ffn + [
    {'name': f"{r['attn_path']}.{r['proj']}", 'cr': r['cr']}
    for r in replaced_attn_detail
]

total_orig_params = sum(r.get('in', 0) * r.get('out', 0) for r in replaced_ffn)
total_monarch_params = 0
for r in replaced_ffn:
    try:
        ml = get_module(model, r['name'])
    except Exception:
        continue
    if isinstance(ml, MonarchRectangular):
        total_monarch_params += ml.structured_parameter_count()

print('=' * 60)
print('FFN 압축 요약')
print(f'  원본 파라미터  : {total_orig_params/1e6:.2f}M')
print(f'  Monarch 파라미터: {total_monarch_params/1e6:.2f}M')
if total_orig_params > 0 and total_monarch_params > 0:
    print(f'  실제 압축률    : {total_orig_params/total_monarch_params:.2f}x')

if not CONVERSION_CACHE_LOADED:
    save_converted_cache(model, replaced_ffn, replaced_attn_detail, all_replaced)
else:
    print(f"Converted cache reused: {CONVERTED_CACHE_DIR}")

# Dispatch only after all module mutation/replacement is complete.
model = dispatch_for_training(model)
model.eval()
print("hf_device_map after dispatch:", getattr(model, "hf_device_map", None))
print("input device after dispatch:", model_input_device(model))
print_cuda_memory("after post-replacement dispatch")

# Do NOT call generate() here.  On the current Transformers/Accelerate stack,
# generation on the sharded Idefics3/SmolVLM model can trigger a CUDA device-side
# assert inside image preprocessing, poisoning the kernel before training.  The
# actual training/evaluation cells should be run after dispatch; generation QA is
# deferred to the validation section or a fresh process.
print("교체 후 generation smoke test: skipped intentionally to avoid CUDA device-side assert before training")


FFN 압축 요약
  원본 파라미터  : 81.40M
  Monarch 파라미터: 42.06M
  실제 압축률    : 1.94x
Converted cache reused: /users_gpu/24_ghangmin/CourseWork/2026_01_semi/project/vlm/monarch_checkpoints/converted_monarch_b679c67f81f8
Moving converted model to cuda:2
hf_device_map after dispatch: None
input device after dispatch: cuda:2
after post-replacement dispatch
  cuda:2: used=1.07 GB free=100.90 GB total=101.97 GB
교체 후 generation smoke test: skipped intentionally to avoid CUDA device-side assert before training


### Disabled stale duplicate cell

This duplicate post-replacement generation smoke cell was disabled because it called `generate()` on the sharded model and could trigger a CUDA device-side assert before training.


## 6. Fine-tuning

**학습 전략**
- **Monarch-only**: 교체된 L, R 행렬만 학습 (나머지 freeze)  
  → 빠른 수렴, 원본 언어 품질 보존
- **Full finetune**: 전체 파라미터 학습 → 더 나은 품질 회복, 더 오래 걸림

**데이터**: LLaVA-Instruct-150K subset 또는 synthetic fallback


In [14]:
# ── 학습 하이퍼파라미터 ────────────────────────────────────────────
TRAIN_MONARCH_ONLY = True     # True: Monarch 파라미터만 / False: 전체
LEARNING_RATE      = 2e-4
NUM_EPOCHS         = 1        # 1 for smoke; 3-5 once runtime is acceptable
BATCH_SIZE         = 4
GRAD_ACCUM_STEPS   = 8        # effective batch = 8 when BATCH_SIZE=1
MAX_SEQ_LEN        = 1280     # SmolVLM image prompts are ~1140 tokens; 1280 leaves answer room with less activation memory
SAVE_STEPS         = 200
LOG_STEPS          = 50
N_SAMPLES          = 128     # 256 for smoke; 1024-3000 for recovery quality

# All-layer Monarch replacement is optimizer-state heavy.  AdamW keeps two full
# state tensors per trainable parameter and can OOM even with BATCH_SIZE=1.
# Adafactor uses factored states for matrices and is the safer default here.
OPTIMIZER_KIND     = "adafactor"   # "adafactor" recommended for all-Monarch; "adamw" for small sanity runs
USE_GRADIENT_CHECKPOINTING = True
WEIGHT_DECAY       = 0.0           # structured factors are already compressed; avoid extra shrinkage by default


In [15]:
import json, requests
from torch.utils.data import Dataset, DataLoader
from transformers import get_cosine_schedule_with_warmup
from transformers.optimization import Adafactor
import torch.optim as optim

# ── 데이터 로드 ────────────────────────────────────────────────────
dataset = None
DATA_SOURCE = None

# 방법 A: HF 캐시 JSON 직접 로드
CACHE_JSON = (Path.home() / ".cache/huggingface/hub"
              / "datasets--liuhaotian--LLaVA-Instruct-150K"
              / "snapshots"
              / "9d451dc7629cfe0469f6ae4432b765cd603d5fcb"
              / "llava_instruct_150k.json")

if CACHE_JSON.exists():
    print(f"캐시 JSON 발견: {CACHE_JSON}")
    try:
        from datasets import load_dataset
        raw = load_dataset("json", data_files=str(CACHE_JSON),
                           split=f"train[:{N_SAMPLES}]")
        dataset    = raw
        DATA_SOURCE = "llava_instruct"
        print(f"로드 완료: {len(dataset)} samples")
    except Exception as e:
        print(f"datasets 로드 실패: {e}")

# 방법 B: HF에서 직접 다운로드
if dataset is None:
    try:
        from datasets import load_dataset
        raw = load_dataset(
            "json",
            data_files="hf://datasets/liuhaotian/LLaVA-Instruct-150K/llava_instruct_150k.json",
            split=f"train[:{N_SAMPLES}]",
        )
        dataset    = raw
        DATA_SOURCE = "llava_instruct"
        print(f"HF 다운로드 완료: {len(dataset)} samples")
    except Exception as e:
        print(f"HF 다운로드 실패: {e}")

# 방법 C: Synthetic fallback
if dataset is None:
    print("Synthetic fallback 사용 (파이프라인 통과 확인용)")
    DATA_SOURCE = "synthetic"

print(f"\nDATA_SOURCE = {DATA_SOURCE}")


캐시 JSON 발견: /home/24_ghangmin/.cache/huggingface/hub/datasets--liuhaotian--LLaVA-Instruct-150K/snapshots/9d451dc7629cfe0469f6ae4432b765cd603d5fcb/llava_instruct_150k.json
로드 완료: 128 samples

DATA_SOURCE = llava_instruct


In [16]:
from PIL import ImageDraw

# ── VQA Dataset ────────────────────────────────────────────────────
class VQADataset(Dataset):
    def __init__(self, data, data_source="llava_instruct"):
        self.data        = data
        self.data_source = data_source

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        if self.data_source == "llava_instruct":
            convs = item.get("conversations", [])
            q = next((c["value"] for c in convs if c["from"] == "human"), "")
            a = next((c["value"] for c in convs if c["from"] == "gpt"),   "")
            q = q.replace("<image>", "").strip()
            img_data = item.get("image")
            if img_data is None:
                img = Image.new("RGB", (256, 256), "gray")
            elif isinstance(img_data, str):
                try:    img = Image.open(img_data).convert("RGB")
                except: img = Image.new("RGB", (256, 256), "gray")
            else:
                img = img_data.convert("RGB")
        elif self.data_source == "synthetic":
            img = item["img"]
            q, a = item["question"], item["answer"]
        else:
            img = Image.new("RGB", (256, 256), "gray")
            q, a = "", ""
        return {"image": img, "question": q, "answer": a}

def collate_fn(batch):
    return {
        "images":    [b["image"]    for b in batch],
        "questions": [b["question"] for b in batch],
        "answers":   [b["answer"]   for b in batch],
    }

# ── Synthetic 데이터 생성 ──────────────────────────────────────────
if DATA_SOURCE == "synthetic":
    synth = []
    for color in ["red", "blue", "green", "yellow"]:
        img = Image.new("RGB", (256, 256), color)
        synth.append({"img": img, "question": "What color is this?", "answer": color})
    for digit in range(1, 11):
        img  = Image.new("RGB", (256, 256), "white")
        draw = ImageDraw.Draw(img)
        draw.text((80, 80), str(digit), fill="black")
        synth.append({"img": img, "question": "What number is shown?", "answer": str(digit)})
    train_ds    = VQADataset(synth, "synthetic")
    print(f"Synthetic: {len(train_ds)} samples")
else:
    train_ds = VQADataset(dataset, DATA_SOURCE)
    print(f"학습 데이터: {len(train_ds)} samples")

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, collate_fn=collate_fn,
)
print(f"배치 수: {len(train_loader)}")


학습 데이터: 128 samples
배치 수: 32


In [17]:
# ── Freeze 설정 ────────────────────────────────────────────────────
if TRAIN_MONARCH_ONLY:
    # MonarchLinear 파라미터만 학습.  Dense base weights stay frozen.
    named_modules_map = dict(model.named_modules())
    for pname, param in model.named_parameters():
        parent_path = ".".join(pname.split(".")[:-1])
        parent_mod  = named_modules_map.get(parent_path)
        param.requires_grad_(isinstance(parent_mod, (MonarchLinear, MonarchRectangular, MonarchSquare)))
else:
    for p in model.parameters():
        p.requires_grad_(True)

# Activation memory, not batch size, is often the next bottleneck after all-layer
# replacement.  Gradient checkpointing trades compute for memory.
if USE_GRADIENT_CHECKPOINTING:
    try:
        model.config.use_cache = False
    except Exception:
        pass
    enabled = False
    for obj in [model, getattr(model, "model", None), getattr(getattr(model, "model", None), "text_model", None), getattr(getattr(model, "model", None), "vision_model", None)]:
        if obj is not None and hasattr(obj, "gradient_checkpointing_enable"):
            try:
                obj.gradient_checkpointing_enable()
                enabled = True
            except Exception as e:
                print(f"[warn] gradient_checkpointing_enable failed on {type(obj).__name__}: {e}")
    print("gradient checkpointing:", "enabled" if enabled else "not available")

trainable_params = [p for p in model.parameters() if p.requires_grad]
trainable = sum(p.numel() for p in trainable_params)
total     = sum(p.numel() for p in model.parameters())
mode_str  = "Monarch-only" if TRAIN_MONARCH_ONLY else "Full"
print(f"{mode_str} 학습: {trainable/1e6:.2f}M / {total/1e6:.2f}M params "
      f"({100*trainable/total:.1f}%)")
if not trainable_params:
    raise RuntimeError("No trainable parameters. Check replacement cells and TRAIN_MONARCH_ONLY setting.")

by_device = {}
for p in trainable_params:
    by_device.setdefault(str(p.device), 0)
    by_device[str(p.device)] += p.numel()
print("trainable params by device:")
for dev, n in sorted(by_device.items()):
    print(f"  {dev:8s}: {n/1e6:.2f}M")

adamw_state_gb = trainable * 2 * 4 / 1e9
param_grad_gb = trainable * 2 * 2 / 1e9  # bf16 params + bf16/fp16-ish grads, approximate
print(f"rough AdamW extra state would be ~{adamw_state_gb:.2f} GB, before gradients/activations")
print(f"rough param+grad footprint is at least ~{param_grad_gb:.2f} GB, before activations")
print_cuda_memory("CUDA memory after freeze/checkpoint setup")


gradient checkpointing: enabled
Monarch-only 학습: 42.06M / 217.11M params (19.4%)
trainable params by device:
  cuda:2  : 42.06M
rough AdamW extra state would be ~0.34 GB, before gradients/activations
rough param+grad footprint is at least ~0.17 GB, before activations
CUDA memory after freeze/checkpoint setup
  cuda:2: used=1.07 GB free=100.90 GB total=101.97 GB


In [18]:
from tqdm.auto import tqdm

# ── SmolVLM 학습용 loss 계산 (batched) ────────────────────────────
def compute_loss(mdl, proc, images, questions, answers, device, dtype):
    device = model_input_device(mdl)
    """
    Batched SmolVLM loss.

    The earlier version looped over samples inside each batch, so BATCH_SIZE did
    not improve throughput. This version batches processor/image tensors and
    pads text+answer sequences manually while keeping labels only on answer
    tokens.
    """
    messages = [
        [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": q or "Describe this image."},
            ],
        }]
        for q in questions
    ]
    prompt_texts = [proc.apply_chat_template(m, add_generation_prompt=True) for m in messages]
    prompt_enc = proc(text=prompt_texts, images=images, return_tensors="pt", padding=True)

    prompt_ids = prompt_enc["input_ids"]
    prompt_mask = prompt_enc.get("attention_mask", torch.ones_like(prompt_ids))
    pad_id = proc.tokenizer.pad_token_id
    if pad_id is None:
        pad_id = proc.tokenizer.eos_token_id

    rows, labels_rows, keep_indices = [], [], []
    for i, answer in enumerate(answers):
        prompt_len = int(prompt_mask[i].sum().item())
        if prompt_len >= MAX_SEQ_LEN:
            print(f"  [skip] prompt_len={prompt_len} >= MAX_SEQ_LEN={MAX_SEQ_LEN}; increase MAX_SEQ_LEN")
            continue

        ids = prompt_ids[i, :prompt_len]
        max_answer_len = MAX_SEQ_LEN - prompt_len
        ans_ids = proc.tokenizer(
            answer,
            return_tensors="pt",
            add_special_tokens=False,
            truncation=True,
            max_length=max_answer_len,
        )["input_ids"][0]
        eos = torch.tensor([proc.tokenizer.eos_token_id], dtype=torch.long)
        ans_ids = torch.cat([ans_ids, eos], dim=0)[:max_answer_len]

        full = torch.cat([ids, ans_ids], dim=0)[:MAX_SEQ_LEN]
        labels = full.clone()
        labels[:min(prompt_len, labels.numel())] = -100
        rows.append(full)
        labels_rows.append(labels)
        keep_indices.append(i)

    if not rows:
        raise RuntimeError("No valid training samples in this batch; increase MAX_SEQ_LEN or inspect prompts.")

    max_len = max(r.numel() for r in rows)
    input_ids = torch.full((len(rows), max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(rows), max_len), dtype=torch.long)
    labels = torch.full((len(rows), max_len), -100, dtype=torch.long)
    for j, (row, lab) in enumerate(zip(rows, labels_rows)):
        input_ids[j, :row.numel()] = row
        attention_mask[j, :row.numel()] = 1
        labels[j, :lab.numel()] = lab

    extra = {
        k: v[keep_indices].to(device)
        for k, v in prompt_enc.items()
        if k not in ("input_ids", "attention_mask")
    }

    try:
        with torch.amp.autocast("cuda", dtype=dtype, enabled=str(device).startswith("cuda")):
            out = mdl(
                input_ids=input_ids.to(device),
                attention_mask=attention_mask.to(device),
                labels=labels.to(device),
                **extra,
            )
            loss = out.loss
    except torch.cuda.OutOfMemoryError as e:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise RuntimeError(
            "CUDA OOM during batched loss. Batch size is already small; reduce MAX_SEQ_LEN, "
            "keep USE_GRADIENT_CHECKPOINTING=True, use OPTIMIZER_KIND=\"adafactor\", "
            "or train fewer converted layer families per run."
        ) from e

    if loss is None or loss.isnan() or loss.isinf():
        raise RuntimeError(f"Invalid loss: {loss}")
    return loss

# ── Optimizer & Scheduler ─────────────────────────────────────────
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_cuda_memory("CUDA memory before optimizer")
trainable_params = [p for p in model.parameters() if p.requires_grad]
if OPTIMIZER_KIND.lower() == "adafactor":
    optimizer = Adafactor(
        trainable_params,
        lr=LEARNING_RATE,
        relative_step=False,
        scale_parameter=False,
        warmup_init=False,
        weight_decay=WEIGHT_DECAY,
    )
elif OPTIMIZER_KIND.lower() == "adamw":
    optimizer = optim.AdamW(
        trainable_params,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
else:
    raise ValueError(f"unknown OPTIMIZER_KIND={OPTIMIZER_KIND!r}")
print(f"optimizer: {OPTIMIZER_KIND}, trainable tensors: {len(trainable_params)}")
total_steps = max(1, len(train_loader) * NUM_EPOCHS // GRAD_ACCUM_STEPS)
warmup_steps = max(1, total_steps // 10)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))

print(f"총 스텝: {total_steps}, 웜업: {warmup_steps}")
print("학습 시작...")


CUDA memory before optimizer
  cuda:2: used=1.07 GB free=100.90 GB total=101.97 GB
optimizer: adafactor, trainable tensors: 156
총 스텝: 4, 웜업: 1
학습 시작...


In [19]:
train_log   = []
global_step = 0
model.train()

for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for step, batch in enumerate(pbar):
        images    = batch["images"]
        questions = batch["questions"]
        answers   = batch["answers"]

        loss = compute_loss(model, processor,
                            images, questions, answers,
                            DEVICE, DTYPE)

        (loss / GRAD_ACCUM_STEPS).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % LOG_STEPS == 0:
                lr_now = scheduler.get_last_lr()[0]
                train_log.append((global_step, loss.item()))
                pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr_now:.2e}")

            if global_step > 0 and global_step % SAVE_STEPS == 0:
                ckpt_path = CKPT_DIR / f"step_{global_step}"
                model.save_pretrained(str(ckpt_path))
                processor.save_pretrained(str(ckpt_path))
                print(f"\n[저장] {ckpt_path}")

print("\n학습 완료!")


Epoch 1/1: 100%|██████████| 32/32 [25:35<00:00, 47.99s/it]


학습 완료!


## 7. 사후 검증

In [20]:
# ── (A) 레이어 품질: Dense vs Monarch cosine similarity ────────────
model.eval()
model_orig.eval()
layer_quality = []

with torch.no_grad():
    for r in replaced_ffn:
        name = r['name']
        try:
            orig_mod = get_module(model_orig, name)
            monarch_mod = get_module(model, name)
        except AttributeError:
            continue
        if not isinstance(orig_mod, nn.Linear):
            continue
        in_f = orig_mod.in_features
        orig_dtype = orig_mod.weight.dtype
        orig_device = orig_mod.weight.device
        monarch_dtype = next(monarch_mod.parameters()).dtype
        monarch_device = next(monarch_mod.parameters()).device

        x_base = torch.randn(32, in_f, device="cpu", dtype=torch.float32)
        y_orig = orig_mod(x_base.to(orig_device, dtype=orig_dtype)).float().cpu()
        y_monarch = monarch_mod(x_base.to(monarch_device, dtype=monarch_dtype)).float().cpu()
        cos = F.cosine_similarity(y_orig, y_monarch, dim=-1).mean().item()
        rel = ((y_orig - y_monarch).norm() / (y_orig.norm() + 1e-8)).item()
        layer_quality.append({'name': name, 'cosine': cos, 'rel_error': rel, 'cr': r['cr']})

print(f"{'레이어':55s} {'cosine':>8} {'rel_err':>8} {'CR':>6}")
print('-' * 82)
for q in layer_quality:
    flag = '✓' if q['cosine'] > 0.99 else ('△' if q['cosine'] > 0.95 else '✗')
    print(f"{q['name']:55s} {q['cosine']:8.4f} {q['rel_error']:8.4f} {q['cr']:6.2f}x {flag}")

import json
with open(CKPT_DIR / 'layer_quality.json', 'w') as f:
    json.dump(layer_quality, f, indent=2)
print(f"\n레이어 품질 저장: {CKPT_DIR / 'layer_quality.json'}")


레이어                                                       cosine  rel_err     CR
----------------------------------------------------------------------------------
model.vision_model.encoder.layers.0.mlp.fc1               0.9856   0.1688   1.60x △
model.vision_model.encoder.layers.1.mlp.fc1               0.9536   0.3006   1.60x △
model.vision_model.encoder.layers.2.mlp.fc1               0.9549   0.2961   1.60x △
model.vision_model.encoder.layers.3.mlp.fc1               0.9523   0.3046   1.60x △
model.vision_model.encoder.layers.4.mlp.fc1               0.9568   0.2886   1.60x △
model.vision_model.encoder.layers.5.mlp.fc1               0.9590   0.2813   1.60x △
model.vision_model.encoder.layers.6.mlp.fc1               0.9583   0.2840   1.60x △
model.vision_model.encoder.layers.7.mlp.fc1               0.9564   0.2894   1.60x △
model.vision_model.encoder.layers.8.mlp.fc1               0.9602   0.2769   1.60x △
model.vision_model.encoder.layers.9.mlp.fc1               0.9556   0.2891   1.60

In [21]:
# ── (B) Regression inference 비교 ─────────────────────────────────
reg_qa = [
    {"image": test_image, "question": "Describe this image in detail.",
     "keywords": ["cat", "animal", "fur", "eyes"]},
    {"image": test_image, "question": "What is the main subject?",
     "keywords": ["cat"]},
    {"image": test_image, "question": "What color is the cat?",
     "keywords": ["brown", "orange", "gray", "grey"]},
]

def run_regression(mdl, qa_list, label):
    mdl.eval()
    results = []
    with torch.inference_mode():
        for item in tqdm(qa_list, desc=label):
            ans    = smolvlm_query(mdl, processor, item["image"],
                                   item["question"], max_new_tokens=64)
            kw_hit = any(k.lower() in ans.lower() for k in item["keywords"])
            results.append({"question": item["question"], "answer": ans,
                            "keywords": item["keywords"], "pass": kw_hit})
    score = sum(r["pass"] for r in results) / len(results)
    print(f"[{label}] {sum(r['pass'] for r in results)}/{len(results)} "
          f"({100*score:.0f}%)")
    for r in results:
        flag = "✓" if r["pass"] else "✗"
        print(f"  {flag} Q: {r['question'][:50]}")
        print(f"    A: {r['answer'][:80]}")
    return score, results

score_orig,    res_orig    = run_regression(model_orig, reg_qa, "원본 Dense")
monarch_eval_label = "Monarch current model (replacement/fine-tune state in this kernel)"
score_monarch, res_monarch = run_regression(model,      reg_qa, monarch_eval_label)


원본 Dense: 100%|██████████| 3/3 [02:14<00:00, 44.89s/it]


[원본 Dense] 3/3 (100%)
  ✓ Q: Describe this image in detail.
    A: The image depicts a close-up view of a cat sitting on a paved surface. The cat i
  ✓ Q: What is the main subject?
    A: There is a cat in the image.
  ✓ Q: What color is the cat?
    A: The cat is orange in color.


Monarch current model (replacement/fine-tune state in this kernel): 100%|██████████| 3/3 [00:12<00:00,  4.22s/it]

[Monarch current model (replacement/fine-tune state in this kernel)] 0/3 (0%)
  ✗ Q: Describe this image in detail.
    A: ,, a:
 a,
 or or or or or or or or or or or or a a this this or or or or or or w
  ✗ Q: What is the main subject?
    A: on a





 on this this a a, of of a



, a a

,,,,,,,, a,,,,,

,,,,
  ✗ Q: What color is the cat?
    A: ,,


In [22]:
# ── (C) 전체 압축률 요약 ──────────────────────────────────────────
total_orig_p = sum(p.numel() for p in model_orig.parameters())
total_monarch_p = sum(p.numel() for p in model.parameters())

monarch_only_p = 0
for _, m in model.named_modules():
    if isinstance(m, MonarchRectangular):
        monarch_only_p += m.structured_parameter_count()

print('=' * 60)
print('압축률 요약')
print('=' * 60)
print(f'  원본 총 파라미터 : {total_orig_p/1e6:.2f}M')
print(f'  Monarch 총 파라미터: {total_monarch_p/1e6:.2f}M')
print(f'  교체된 Monarch 파라미터: {monarch_only_p/1e6:.2f}M')
print(f'  FFN 교체 레이어 수: {len(replaced_ffn)}')
print(f'  Attn 교체 projection 수: {len(replaced_attn_detail)}')
print()
print(f'  Regression 스코어 (원본)   : {100*score_orig:.0f}%')
print(f'  Regression 스코어 (Monarch): {100*score_monarch:.0f}%')
print()

final_path = CKPT_DIR / 'final'
model.save_pretrained(str(final_path))
processor.save_pretrained(str(final_path))
print(f'최종 체크포인트 저장: {final_path}')


압축률 요약
  원본 총 파라미터 : 256.48M
  Monarch 총 파라미터: 217.11M
  교체된 Monarch 파라미터: 42.06M
  FFN 교체 레이어 수: 72
  Attn 교체 projection 수: 0

  Regression 스코어 (원본)   : 100%
  Regression 스코어 (Monarch): 0%



Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.53s/it]

최종 체크포인트 저장: /users_gpu/24_ghangmin/CourseWork/2026_01_semi/project/vlm/monarch_checkpoints/final


In [23]:
# ── (D) 학습 Loss 곡선 ────────────────────────────────────────────
if train_log:
    try:
        import matplotlib.pyplot as plt
        steps, losses = zip(*train_log)
        plt.figure(figsize=(8, 4))
        plt.plot(steps, losses, marker="o", markersize=3)
        plt.xlabel("Step"); plt.ylabel("Loss")
        plt.title("Monarch Fine-tuning Loss Curve")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(str(CKPT_DIR / "loss_curve.png"), dpi=120)
        plt.show()
        print(f"Loss 곡선 저장: {CKPT_DIR / 'loss_curve.png'}")
    except ImportError:
        print("matplotlib 없음 — pip install matplotlib")
else:
    print("학습 로그 없음 (학습이 실행되지 않았거나 LOG_STEPS보다 스텝이 적음)")


학습 로그 없음 (학습이 실행되지 않았거나 LOG_STEPS보다 스텝이 적음)
